# 02. D2: 사람 작성·주석 절차

D2는 캐시 **안전성**의 주 실험이다. 원질문 300그룹(calibration 100 / test 200)마다 사람이 **같은 뜻의 다른 표현 1개**와 **뜻이 달라지는 함정 질문 1개**를 쓰고, 두 사람이 따로 판정한다.

## A. 작성 규칙 (작성자)

`d2/authoring-sheet.csv`를 복사해 채운다. 원본 시트는 고치지 않는다.

- `paraphrase_text`: 원질문과 **정답이 같아야** 한다. 객관식은 선택지 문장과 순서를 그대로 두고 질문 줄기만 바꾼다. 공백만 바꾼 표현은 거부된다.
- `hard_negative_text`: 표면은 비슷하지만 **저장된 답이 틀리게 되도록** 바꾼다. 시트의 `negative_type`을 따른다.
  - `negation` 부정 추가·제거 · `entity_swap` 개체 교체 · `number_unit` 숫자·단위 · `time` 시점 · `order` 출발/도착·순서 교체 · `code_language` 코드 언어 · `output_format` 출력 형식
- 변형은 BGE 기준 512토큰 이하여야 한다. 서비스가 그보다 긴 질문을 거절하므로 캐시에 닿을 수 없다. 검증 단계에서 거부된다.
- `author`: 작성자 이름. 원질문 정답(`gold`)과 저장 답(`cached_answer`)은 참고용이다.

## B. 판정 규칙 (주석자 2명, 서로 보지 않고)

`d2/annotation-A.csv`, `annotation-B.csv`를 각자 채운다. 시트에는 작성 의도(표현/함정)가 숨겨져 있고 순서도 섞여 있다.

- `semantic_equivalent`: 변형 질문이 원질문과 같은 것을 묻는가 `Y`/`N`
- `cached_answer_correct_for_target`: 원질문에 저장된 답(`cached_answer`)을 **변형 질문의 답으로 내보내도 맞는가** `Y`/`N`/`U`(판단 불가). 같은 뜻이어도 원답 자체가 틀렸으면 `N`이다.
- `rationale`: 한 줄 근거. `annotator`: 판정자 이름(두 시트가 달라야 한다).

## C. 조정

두 사람이 다른 항목은 `d2/adjudication-needed.csv`로 나온다. 제3자 또는 두 사람 합의로 `final`, `reason`, `adjudicator`를 채운다. 원판정은 그대로 보존된다.

모든 파일은 UTF-8 CSV다(Excel·Google Sheets에서 열고 CSV로 다시 저장).

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

RUN_ID = os.environ.get("CACHE_EXP_RUN_ID", "run001")  # 새 계획·재시작이면 새 ID
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LLM_ROUTER = Path("/content/drive/MyDrive/LLM_Router")
    KIT = LLM_ROUTER / "cache-experiment"
    zip_path = KIT / "source/cache-source.zip"
    expected = (KIT / "source/cache-source.sha256").read_text().split()[0]
    assert hashlib.sha256(zip_path.read_bytes()).hexdigest() == expected, "source ZIP SHA mismatch"
    REPO = Path("/content/cache-source")
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
        z.extractall(REPO)
    for name, digest in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
        assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
    HERE = REPO / "ml/notebooks/cache_experiment"
    DATA_ROOT, OUTPUT_ROOT = KIT / "data-root", KIT / "runs"
    ROUTER_DIR = LLM_ROUTER / "transfer-experiments/inputs/experiment-A/models/r0"
    os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(LLM_ROUTER / "retraining-effect/logger/logger-log.parquet"))
    assert sys.version_info[:2] in {(3, 11), (3, 12)}, "런타임 버전 2026.07(Python 3.12)을 선택하세요"
    if not (REPO / ".installed").exists():
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1",
                        "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(HERE / "requirements-colab.txt")], check=True)
        (REPO / ".installed").write_text("ok")
        # Colab already imported its own numpy; mixing it with the pinned one breaks
        # transformers. Restart the runtime (files in /content survive) and run all again.
        print("설치 완료. 런타임을 재시작합니다. 재시작되면 '모두 실행'을 다시 누르세요.")
        os.kill(os.getpid(), 9)
    import numpy
    pinned = next(l.split("==")[1].strip() for l in (HERE / "requirements-colab.txt").read_text().splitlines()
                  if l.startswith("numpy=="))
    assert numpy.__version__ == pinned, (
        f"numpy {numpy.__version__} != {pinned}: 런타임 → 세션 다시 시작 후 '모두 실행'을 다시 누르세요")
else:
    # 로컬: 저장소 루트의 .venv-cache 커널. 입력은 ml/data·ml/artifacts가 있는 체크아웃,
    # 산출물은 Drive 동기화 폴더의 kit runs/ 가 있으면 그곳(Colab과 같은 run을 이어 쓴다).
    HERE = Path.cwd()
    DATA_ROOT = Path(os.environ.get("CACHE_EXP_DATA_ROOT", HERE.parents[2]))
    synced = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/cache-experiment/runs"))
    OUTPUT_ROOT = os.environ.get("CACHE_EXP_OUTPUT_ROOT") or (synced[0] if synced else None)
    ROUTER_DIR = None  # config router.package_path
    found = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/retraining-effect/logger/logger-log.parquet"))
    if found:
        os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(found[0]))
CONFIG = Path(os.environ.get("CACHE_EXP_CONFIG", HERE / "experiment_config.json"))
sys.path.insert(0, str(HERE))

import pandas as pd
import cache_experiment as ce
import models
import pipeline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
run = pipeline.Run(DATA_ROOT, RUN_ID, CONFIG, OUTPUT_ROOT)
print("run directory:", run.root)

In [ ]:
runtime = models.runtime_versions(run.config)
embedder = models.Embedder(run.config, device="cpu")  # 변형 질문 토큰 수 확인용(토크나이저만 쓴다)

## 1. 작성 시트 검증과 주석 시트 생성

작성 완료본을 `runs/<RUN_ID>/d2/authoring-filled-input.csv`로 저장한다(다른 경로면 `FILLED_AUTHORING`을 바꾼다). 문제가 없으면 변형 600개를 봉인하고 빈 주석 시트 두 개를 만든다.

In [ ]:
FILLED_AUTHORING = run.path("d2", "authoring-filled-input.csv")  # 작성 완료본 경로
problems = pipeline.d2_validate_authoring(run, FILLED_AUTHORING, embedder)
problems

In [ ]:
if problems.empty:
    variants = pipeline.d2_make_annotation_sheets(run, FILLED_AUTHORING, embedder)
    print(variants.groupby(["d2_split", "role"]).size())
    print("annotation sheets:", run.path("d2", "annotation-A.csv"), run.path("d2", "annotation-B.csv"))

## 2. 주석 병합과 일치도

두 사람의 시트를 받으면 실행한다. 불일치가 있으면 조정 시트를 만들고 멈춘다. 조정 파일을 채운 뒤 `ADJUDICATION`을 지정해 다시 실행하면 최종 라벨을 봉인한다.

In [ ]:
FILLED_A = run.path("d2", "annotation-A-input.csv")
FILLED_B = run.path("d2", "annotation-B-input.csv")
ADJUDICATION = None  # 예: run.path("d2", "adjudication-input.csv")
agreement, pending = pipeline.d2_merge_annotations(run, FILLED_A, FILLED_B, ADJUDICATION)
agreement, len(pending)